# exp146 tvt_plus_z_beam_smoothness_penalty train

Train-side pseudo-tail audit for Beam search variants whose transition cost includes `U = TVT + Z - (T0 + Z0)` and `dU/dMD = dTVT/dMD + dZ/dMD` penalties. This is separate from exp142 trajectory-aware PF.

## 1. Setup and configuration


In [ ]:
import json
from pathlib import Path

import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from tvt_plus_z_beam_smoothness_penalty import (
    EXP072_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    run_audit,
)

config = load_config()
paths = ExperimentPaths()
print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('status:', get_nested(config, 'experiment.status'))
print('parent:', get_nested(config, 'lineage.parent'))
print('implementation:', get_nested(config, 'lineage.implementation_source'))
print('artifacts dir:', paths.artifacts_dir)


## 2. Input preview


In [ ]:
cache_path = find_artifact(
    EXP072_TRAIN_FEATURES,
    get_nested(config, 'data.exp072_train_feature_cache_local'),
)
print('exp072 cache:', cache_path)
header = pd.read_csv(cache_path, nrows=0).columns.tolist()
required = ['id', 'well', 'target', 'last_known_tvt', 'md_since', 'beam_mean_d', 'likpf_mean_d', 'pf_ancc', 'pf_z']
print('missing required:', [col for col in required if col not in header])
display(pd.read_csv(cache_path, usecols=[col for col in required if col in header], nrows=5))


## 3. Beam variants


In [ ]:
variants = get_nested(config, 'model.beam_smoothness.variants') or []
display(pd.DataFrame(variants))
print('primary baseline:', get_nested(config, 'audit.primary_baseline'))
display(pd.DataFrame(get_nested(config, 'audit.candidates') or []))


## 4. Run audit


In [ ]:
summary = run_audit(config)
print(json.dumps(summary, indent=2, sort_keys=True)[:8000])


## 5. Metrics and artifacts


In [ ]:
artifact_dir = paths.artifacts_dir
metrics_path = artifact_dir / f'{OUTPUT_PREFIX}_candidate_metrics.csv'
bucket_path = artifact_dir / f'{OUTPUT_PREFIX}_bucket_metrics.csv'
quality_path = artifact_dir / f'{OUTPUT_PREFIX}_beam_quality.csv'
summary_path = artifact_dir / f'{OUTPUT_PREFIX}_summary.json'

metrics = pd.read_csv(metrics_path)
display(metrics.head(30))
bucket = pd.read_csv(bucket_path)
display(bucket.head(40))
quality = pd.read_csv(quality_path)
display(quality.groupby('variant').agg({
    'well': 'nunique',
    'cost_per_row': 'mean',
    'mean_abs_step': 'mean',
    'mean_abs_du_slope': 'mean',
}).reset_index())
print('artifacts:', sorted(p.name for p in artifact_dir.glob(f'{OUTPUT_PREFIX}*')))
print('summary:', summary_path)
